# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [38]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** item from an order.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [39]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path = 'lines', meta =['order', 'vendor_id']) 
assert len(lines) == 4 and lines['qty'].sum() == 7 
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


Explanation: I flattened the orders so that each row would have a single line item using json_normalize(). I then asserted that there were 4 rows and the quantity total was 7 to ensure my flattening worked.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [40]:
vendorInfo = pd.json_normalize(vendors_json) 
joined = lines.merge(vendorInfo, on = 'vendor_id', how = 'left', indicator = True) 
matchVendor = joined[joined['_merge'] == 'left_only'] 
print(matchVendor[['vendor_id', 'qty']])

  vendor_id  qty
3      V-07    3


Explanation: First I used json_normalize() to create vendorInfo, then joined vendorInfo on 'vendor_id' to match each vendor to its information. I then singled out which vendor didn't have a match and printed it out, revealing it was V-07 with a quantity of 3.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
joined['name'] = joined['name'].fillna('Missing Vendor')
#I decided to keep the unmatched vendor and name it 'Missing Vendor' so that its quantity of 3 is still represented in the table.
joined.groupby('name')['qty'].sum()


name
Cav Merch         1
Hoos Burgers      3
Missing Vendor    3
Name: qty, dtype: int64

Explanation: I renamed the unmatched vendor 'Missing Vendor' so that its quantity data was still represented in the table and then output the table of each vendor with its total quantity.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [42]:
joined.groupby('zone')['qty'].sum()

zone
A    4
Name: qty, dtype: int64

Explanation: I grouped the vendors by zone and output the zone item totals and saw that only zone A got output with 4 items sold. The total is 4 and the difference of 3 belonged to the unmatched vendor that was not designated a zone.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [43]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

for item in ragged_json:
    if 'lines' not in item:
        item['lines'] = [] 
newFrame = pd.json_normalize(ragged_json, record_path = 'lines', meta = ['order', 'vendor_id'])
newFrame

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


Explanation: First I checked every item in ragged_json to see if it was missing the 'lines' key, and if it was I assigned it an empty []. Then I used json_normalize() to create a frame where a missing quantity is represented as 'NaN', and observed that Fries had this quantity label. Order 5 was not present in the table due to it having no lines.

### Q6 — Validate

In [44]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) The "one row per line item" is the useful shape here because it isolates each item to its own row, making total quantity sold per item obvious instead of having a list of different items with different amounts sold per row. The question that becomes easy that would be hard with one row per order is: What is the total quantity sold of this item? 

b) In Q5, the practical difference between a missing quantity and a quantity of zero is a quantity of zero is an actual recording of how many of an item were sold, while a missing quantity means the amount sold is not known and has no number associated to it. If I conflate them then numbers like sums and averages could be broken downstream by potentially deviating from their true values, and the overall data could be inaccurate due to assuming a missing quantity equals zero.